# Export YOLO Model to NCNN

This notebook exports the YOLOv11n-seg model to NCNN format for 2-4x faster inference on Raspberry Pi 4.

**What this does:**
- Loads the PyTorch model (`yolov11n-seg.pt`)
- Exports to NCNN format (optimized for ARM/CPU)
- Creates a `yolov11n-seg_ncnn_model/` directory
- Your detector will auto-detect and use it

**Result:** ~2-4x speedup on Raspberry Pi with no accuracy loss

## Step 1: Check model exists

In [1]:
from pathlib import Path
import os

model_path = Path('../../../../src/models/yolov11n-seg.pt')

print(f"Model path: {model_path.resolve()}")
print(f"Exists: {model_path.exists()}")
print(f"Size: {model_path.stat().st_size / (1024*1024):.1f} MB")

if not model_path.exists():
    print("\nModel not found! Download with:")
    print("  from ultralytics import YOLO")
    print("  YOLO('yolov11n-seg')  # Auto-downloads to ~/.cache/")

Model path: C:\Users\harry\Desktop\autobin-jaft\src\models\yolov11n-seg.pt
Exists: True
Size: 5.9 MB


## Step 2: Load the model

In [2]:
from ultralytics import YOLO

print("Loading model...")
model = YOLO(str(model_path))
print(f"✓ Model loaded: {model.model_name}")
print(f"Task: {model.task}")

Loading model...
✓ Model loaded: ..\..\..\..\src\models\yolov11n-seg.pt
Task: segment


## Step 3: Export to NCNN

This step takes 2-5 minutes depending on your hardware.

In [3]:
import time

print("Exporting to NCNN format...")
print("This may take 2-5 minutes...\n")

start = time.time()
try:
    export_path = model.export(format='ncnn', imgsz=640)
    elapsed = time.time() - start
    print(f"\n✓ Export complete in {elapsed:.1f} seconds")
    print(f"Export path: {export_path}")
except Exception as e:
    print(f"✗ Export failed: {e}")
    print("\nTroubleshooting:")
    print("  - Make sure ultralytics is up to date: pip install -U ultralytics")
    print("  - NCNN export requires onnx: pip install onnx")

Exporting to NCNN format...
This may take 2-5 minutes...

Ultralytics 8.4.82  Python-3.11.2 torch-2.12.1+cpu CPU (12th Gen Intel Core i7-1255U)
WARNING NCNN export does not support end2end models, disabling end2end branch.
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n-seg summary (fused): 113 layers, 2,868,664 parameters, 0 gradients, 9.7 GFLOPs

PyTorch: starting from '..\..\..\..\src\models\yolov11n-seg.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 116, 8400), (1, 32, 160, 160)) (5.9 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
     ---------------------------------------- 5.1/5.1 MB 709.1 kB/s eta 0:00:00

[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

requirements: AutoUpdate success  11.1s
WARNING requirements: Restart runtime or rerun comma

## Step 4: Verify the export

In [4]:
from pathlib import Path

ncnn_dir = Path('../../../../src/models/yolov11n-seg_ncnn_model')

print(f"NCNN export directory: {ncnn_dir}")
print(f"Exists: {ncnn_dir.exists()}")

if ncnn_dir.exists():
    print(f"\nContents:")
    for item in sorted(ncnn_dir.iterdir()):
        size = item.stat().st_size / (1024) if item.is_file() else 0
        size_str = f"{size:.1f} KB" if size > 0 else "(dir)"
        print(f"  {item.name:40s} {size_str}")
    
    print(f"\n✓ NCNN export ready!")
    print(f"\nYour detector will automatically use this when you run:")
    print(f"  python tests/test_ibvs_centering.py --live")
else:
    print("\n✗ NCNN directory not found. Check the export path above.")

NCNN export directory: ..\..\..\..\src\models\yolov11n-seg_ncnn_model
Exists: True

Contents:
  __pycache__                              (dir)
  metadata.yaml                            1.5 KB
  model.ncnn.bin                           11304.6 KB
  model.ncnn.param                         24.6 KB
  model_ncnn.py                            0.8 KB

✓ NCNN export ready!

Your detector will automatically use this when you run:
  python tests/test_ibvs_centering.py --live


## Step 5: Quick inference test (optional)

In [5]:
import numpy as np
import time

# Create a dummy image
dummy_frame = np.zeros((720, 1280, 3), dtype=np.uint8)

print("Testing PyTorch inference (baseline)...")
start = time.time()
result_pt = model.predict(source=dummy_frame, device='cpu', imgsz=640, verbose=False)
pt_time = (time.time() - start) * 1000
print(f"  Time: {pt_time:.1f}ms")

print("\nTesting NCNN inference (if available)...")
try:
    ncnn_model = YOLO(str(ncnn_dir), task='segment')
    start = time.time()
    result_ncnn = ncnn_model.predict(source=dummy_frame, device='cpu', imgsz=640, verbose=False)
    ncnn_time = (time.time() - start) * 1000
    print(f"  Time: {ncnn_time:.1f}ms")
    speedup = pt_time / ncnn_time
    print(f"\n✓ Speedup: {speedup:.1f}x")
except Exception as e:
    print(f"  Not available yet (this is normal if export just finished)")

Testing PyTorch inference (baseline)...
  Time: 290.0ms

Testing NCNN inference (if available)...
Loading ..\..\..\..\src\models\yolov11n-seg_ncnn_model for NCNN inference...
  Time: 278.1ms

✓ Speedup: 1.0x


## Done!

✓ Your NCNN export is ready at:
```
src/models/yolov11n-seg_ncnn_model/
```

**Next steps:**

1. Copy `yolov11n-seg_ncnn_model/` to your Raspberry Pi if you exported elsewhere
2. Run the live demo:
   ```bash
   python tests/test_ibvs_centering.py --live
   ```
3. Watch for:
   ```
   ✓ Using NCNN export: src/models/yolov11n-seg_ncnn_model
   ```
4. Check `[TIMING]` output to see the speedup (should be 2-4x faster)

**If it still uses PyTorch:**
- Check that `yolov11n-seg_ncnn_model/` exists in `src/models/`
- The names must match exactly (including the suffix)
- Restart Python/Jupyter to clear any cached model loads